Below I’ll give you modular PyTorch functions for:

* Real Möbius map ($M_w(x)$) on ball / sphere
* Kuramoto-on-sphere order parameter and vector field
* Reduced hyperbolic ($w$)-dynamics (both via Möbius and via autograd gradient)
* Simple simulation + Plotly visualization hooks for 2D (disk/$S^1$) and 3D (ball/$S^2$)

Everything is real-valued, dot-product based, and autograd-friendly.

# 1. Core math building blocks (PyTorch)

In [1]:
import torch
from torch import Tensor
import math

## 1.1 Basic helpers

In [2]:
def normalize(x: Tensor, eps: float = 1e-9) -> Tensor:
    """
    Normalize vectors along the last dimension.
    x: [..., d]
    returns: [..., d] with unit norm (or zero if input ~0).
    """
    norm = x.norm(dim=-1, keepdim=True)
    return x / (norm + eps)


def dot(a: Tensor, b: Tensor) -> Tensor:
    """
    Dot product along the last dimension.
    a, b: [..., d]
    returns: [...] (scalar per batch entry)
    """
    return (a * b).sum(dim=-1)

## 1.2 Real Möbius map on ball / sphere
This is the real version of the Möbius map ($M_w$) we discussed:

$$
M_w(x) = \frac{(1 - |w|^2)(x - |x|^2 w)}{1 - 2\langle w,x\rangle + |w|^2 |x|^2} - w.
$$

In [3]:
%cd /Users/adamsobieszek/pitch/pitch-website/public/notebooks/kuramoto/LMSSPP/src
from lmsspp import (
    random_points_on_sphere, integrate_w_euler, make_sphere_figure, reconstruct_sphere_trajectory
)
import torch

p3 = random_points_on_sphere(640, 3)
a = torch.ones(640)
res = integrate_w_euler(torch.zeros(3), p3, a, dt=1e-3, steps=300)
x_traj = reconstruct_sphere_trajectory(res.trajectory, p3)
fig = make_sphere_figure(x_traj[-1], w_t=res.trajectory[-1], w_path=res.trajectory)
fig.show()


/Users/adamsobieszek/pitch/pitch-website/public/notebooks/kuramoto/LMSSPP/src


In [4]:
from lmsspp import make_lms_circle_plotly_widget

widget = make_lms_circle_plotly_widget(steps=420, dt=0.012, rng_seed=0)
display(widget.layout)



    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'black', 'wid…

In [5]:
from lmsspp.lms_plotly_widget import LMSCirclePlotlyWidget, SliderSpec

widget = LMSCirclePlotlyWidget(
    slider_specs=[
        SliderSpec("custom_alpha", "Custom alpha", 0.0, 1.0, 0.05, 0.5),
    ]
)
display(widget.layout)


    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'black', 'wid…

In [6]:
from lmsspp import make_lms_ball3d_widget
from IPython.display import display

widget = make_lms_ball3d_widget(w_mode="autograd", rng_seed=0)
display(widget.layout)

import torch
from lmsspp import (
    random_points_on_sphere, skew_symmetric_from_axis, integrate_lms_reduced_euler
)

N, d = 128, 3
base = random_points_on_sphere(N, d=d, dtype=torch.float64)
weights = torch.ones(N, dtype=torch.float64) / N
w0 = torch.tensor([0.05, 0.0, 0.0], dtype=torch.float64)
zeta0 = torch.eye(3, dtype=torch.float64)
A = skew_symmetric_from_axis(torch.tensor([0.,0.,1.], dtype=torch.float64), rate=1.0)

traj = integrate_lms_reduced_euler(
    w0, zeta0, base, weights, A=A, coupling=1.0, dt=5e-2, steps=1000, w_mode="autograd"
)


Box(children=(FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'lightgrey', '…

In [7]:
from lmsspp import make_lms_ball3d_backward_two_sheet_widget
from IPython.display import display

widget = make_lms_ball3d_backward_two_sheet_widget(
    w_mode="explicit",
    rng_seed=0,
    outer_radius_display=1.5,   # camera range for outside sheet
    outer_radius_cap=1.0,       # clip very large radii for readability
    force_backward_time=True,   # native backward dynamics
)
display(widget.layout)


Box(children=(FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'lightgrey', '…

In [8]:
from lmsspp import make_lms_ball3d_hydrodynamic_ensemble_widget
w = make_lms_ball3d_hydrodynamic_ensemble_widget()
w.layout


Box(children=(FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'lightgrey', '…

In [1]:
import sys
from pathlib import Path
from IPython.display import display

root = Path("/Users/adamsobieszek/pitch/pitch-website/public/notebooks/kuramoto/LMSSPP/src")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from lmsspp.simpler_lms_ball3d_widget import LMSBall3DEntropyShellEnsembleWidget

entropy_widget = LMSBall3DEntropyShellEnsembleWidget(
    w_mode="explicit",
    rng_seed=0,
)
display(entropy_widget.layout)


Box(children=(FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'lightgrey', '…

In [11]:

from lmsspp.lms_ball4d_widget import LMSBall4DWidget

entropy_widget = LMSBall4DWidget(
    w_mode="explicit",
    rng_seed=0,
)
display(entropy_widget.layout)


Box(children=(HBox(children=(FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color':…